In [1]:
# Load environment variables and suppress warnings for clean output
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("Model initialized with gpt-5-mini.")

Model initialized with gpt-5-mini.


In [3]:
# Define realistic multi-variable application input
customer = "Northwind Traders"
product_area = "Dashboard"
platform = "Windows laptop"
symptoms = "Loads in 20–30 seconds, sometimes TIMEOUT_504"
timeline = "Started after yesterday's update"

In [4]:
# Build raw string prompt manually using direct formatting
raw_prompt = (
    f"You are a technical support assistant. Keep output short and structured.\n\n"
    f"Customer: {customer}\n"
    f"Product area: {product_area}\n"
    f"Platform: {platform}\n\n"
    f"Symptoms: {symptoms}\n"
    f"Timeline: {timeline}\n\n"
    f"Return exactly three lines:\n"
    f"Severity: <Low|Medium|High>\n"
    f"Summary: <one line>\n"
    f"Next: <one action>"
)

print("=== RAW STRING PROMPT ===")
print(raw_prompt)

=== RAW STRING PROMPT ===
You are a technical support assistant. Keep output short and structured.

Customer: Northwind Traders
Product area: Dashboard
Platform: Windows laptop

Symptoms: Loads in 20–30 seconds, sometimes TIMEOUT_504
Timeline: Started after yesterday's update

Return exactly three lines:
Severity: <Low|Medium|High>
Summary: <one line>
Next: <one action>


In [5]:
# Define shared instructions so both prompts ask for the same output
system_instruction = "You are a technical support assistant. Keep output short and structured."

human_instruction = (
    "Customer: {customer}\n"
    "Product area: {product_area}\n"
    "Platform: {platform}\n\n"
    "Symptoms: {symptoms}\n"
    "Timeline: {timeline}\n\n"
    "Return exactly three lines:\n"
    "Severity: <Low|Medium|High>\n"
    "Summary: <one line>\n"
    "Next: <one action>"
)

# Build ChatPromptTemplate (placeholders preserved)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", human_instruction),
])

print("\n=== CHAT PROMPT TEMPLATE DEFINITION ===")
print(chat_prompt)


=== CHAT PROMPT TEMPLATE DEFINITION ===
input_variables=['customer', 'platform', 'product_area', 'symptoms', 'timeline'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a technical support assistant. Keep output short and structured.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['customer', 'platform', 'product_area', 'symptoms', 'timeline'], input_types={}, partial_variables={}, template='Customer: {customer}\nProduct area: {product_area}\nPlatform: {platform}\n\nSymptoms: {symptoms}\nTimeline: {timeline}\n\nReturn exactly three lines:\nSeverity: <Low|Medium|High>\nSummary: <one line>\nNext: <one action>'), additional_kwargs={})]


In [6]:
# Execute raw string prompt directly
raw_response = model.invoke(raw_prompt).content

print("=== RAW STRING OUTPUT ===")
print(raw_response)

# Execute ChatPromptTemplate via LCEL
chain = chat_prompt | model
chat_response = chain.invoke({
    "customer": customer,
    "product_area": product_area,
    "platform": platform,
    "symptoms": symptoms,
    "timeline": timeline
}).content

print("\n=== CHAT PROMPT TEMPLATE OUTPUT ===")
print(chat_response)

=== RAW STRING OUTPUT ===
Severity: High
Summary: Dashboard began loading very slowly (20–30s) and intermittently returning TIMEOUT_504 after yesterday's update on Windows laptops.
Next: Roll back yesterday's dashboard update in production immediately.

=== CHAT PROMPT TEMPLATE OUTPUT ===
Severity: High
Summary: Dashboard loads slowly (20–30s) and intermittently returns 504 TIMEOUT after yesterday's update on Windows laptops
Next: Roll back yesterday's Dashboard update immediately and monitor response times


In [7]:
# Reuse structured template with new input
new_input = {
    "customer": "Contoso Ltd",
    "product_area": "Billing",
    "platform": "Web",
    "symptoms": "Incorrect invoice totals",
    "timeline": "After pricing configuration change"
}

chat_response_2 = chain.invoke(new_input).content
print(chat_response_2)

Severity: High
Summary: Invoice totals are incorrect after the recent pricing configuration change; calculations (discounts/taxes) no longer match expected results.
Next: Revert the pricing configuration to the last known-good settings and run a targeted invoice recalculation for affected accounts.
